# 📈 Evaluating Fine-Tuned Models

**Measure and improve fine-tuned model performance**

---

## 📋 Overview

**What you'll learn:**
- Evaluation metrics for fine-tuning
- Creating test sets
- A/B testing models
- Detecting overfitting
- Continuous evaluation

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
import json
import numpy as np
from typing import List, Dict
from collections import defaultdict
import pandas as pd

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Evaluate?

### Don't Trust Training Loss!

```
Training loss: 0.1  ← Looks great!
Validation loss: 0.5  ← Overfitting!
Production accuracy: 60%  ← Not usable 😞
```

### What to Measure:

**1. Quantitative Metrics**
- Accuracy
- F1 score
- BLEU/ROUGE (for generation)
- Perplexity

**2. Qualitative Metrics**
- Response quality
- Tone/style consistency
- Factual accuracy
- Human preference

**3. Business Metrics**
- User satisfaction
- Task completion rate
- Response time
- Cost per request

### Evaluation Workflow:

```
1. Create test set (held-out data)
2. Run model on test set
3. Calculate metrics
4. Compare to baseline
5. A/B test in production
```

## 📊 Creating a Test Set

In [ ]:
# Create test set with expected outputs
test_set = [
    {
        "input": "How do I reset my password?",
        "expected_output": "To reset your password: 1. Go to login page 2. Click 'Forgot Password' 3. Enter your email 4. Check inbox for reset link",
        "category": "account"
    },
    {
        "input": "Where is my order?",
        "expected_output": "Track your order: 1. Log into account 2. Go to 'My Orders' 3. Click order number for tracking",
        "category": "shipping"
    },
    {
        "input": "Can I return this item?",
        "expected_output": "Yes! 30-day return policy. Items must be unused in original packaging. Visit 'Returns' in your account to start.",
        "category": "returns"
    },
    {
        "input": "Do you ship internationally?",
        "expected_output": "Yes, we ship to 50+ countries. Delivery in 7-14 business days. Customs fees may apply.",
        "category": "shipping"
    },
    {
        "input": "The product arrived damaged",
        "expected_output": "I'm sorry! Please provide: order number, photo of damage, and brief description. We'll send a replacement or refund.",
        "category": "support"
    },
]

print(f"📊 Test Set: {len(test_set)} examples\n")
print("Categories:")
categories = defaultdict(int)
for example in test_set:
    categories[example['category']] += 1

for cat, count in categories.items():
    print(f"  {cat}: {count}")

## 🎯 Exact Match Accuracy

In [ ]:
def calculate_exact_match(predictions: List[str], expected: List[str]) -> float:
    """Calculate exact match accuracy."""
    
    matches = sum(1 for pred, exp in zip(predictions, expected) if pred.strip().lower() == exp.strip().lower())
    
    return matches / len(predictions)

def calculate_contains_match(predictions: List[str], expected: List[str]) -> float:
    """Check if expected keywords are in prediction."""
    
    matches = 0
    for pred, exp in zip(predictions, expected):
        # Extract key phrases from expected
        exp_words = set(exp.lower().split())
        pred_words = set(pred.lower().split())
        
        # Check overlap
        overlap = len(exp_words & pred_words) / len(exp_words)
        
        if overlap > 0.5:  # 50% word overlap
            matches += 1
    
    return matches / len(predictions)

# Example
predictions = [
    "To reset: go to login, click forgot password, enter email",
    "Track order in My Orders section",
    "Yes, 30-day returns accepted"
]

expected = [
    "To reset your password: go to login page, click 'Forgot Password', enter your email",
    "Track your order in 'My Orders'",
    "Yes! 30-day return policy"
]

exact = calculate_exact_match(predictions, expected)
contains = calculate_contains_match(predictions, expected)

print("📊 Accuracy Metrics:")
print(f"   Exact match: {exact*100:.1f}%")
print(f"   Contains match: {contains*100:.1f}%")
print("\n💡 For text generation, contains_match is more realistic")

## 🤖 LLM-as-Judge Evaluation

In [ ]:
def llm_judge_evaluation(
    input_text: str,
    prediction: str,
    expected: str,
    criteria: List[str] = None
) -> Dict:
    """
    Use LLM to judge response quality.
    
    Args:
        input_text: User query
        prediction: Model's response
        expected: Expected/reference response
        criteria: List of evaluation criteria
    """
    
    if criteria is None:
        criteria = [
            "Accuracy: Does it answer the question correctly?",
            "Completeness: Is all necessary information included?",
            "Clarity: Is it easy to understand?",
            "Tone: Is it professional and helpful?"
        ]
    
    criteria_text = "\n".join(f"- {c}" for c in criteria)
    
    prompt = f"""You are evaluating a customer support response.

User Question: {input_text}

Model Response: {prediction}

Reference Response: {expected}

Evaluate the Model Response on these criteria:
{criteria_text}

For each criterion, give a score from 1-5 (5 = excellent).

Respond in JSON format:
{{
  "accuracy": <score>,
  "completeness": <score>,
  "clarity": <score>,
  "tone": <score>,
  "overall": <average score>,
  "reasoning": "<brief explanation>"
}}"""
    
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    
    try:
        result = json.loads(response.choices[0].message.content)
        return result
    except:
        return {"error": "Failed to parse evaluation"}

# Example evaluation
print("🤖 LLM-as-Judge Example\n")

evaluation_example = {
    "input": "How do I reset my password?",
    "prediction": "Go to the login page and click 'Forgot Password'. Enter your email and you'll get a reset link.",
    "expected": "To reset your password: 1. Go to login page 2. Click 'Forgot Password' 3. Enter your email 4. Check inbox"
}

print("💡 In production, you would call:")
print("""
result = llm_judge_evaluation(
    input_text=evaluation_example['input'],
    prediction=evaluation_example['prediction'],
    expected=evaluation_example['expected']
)

Example output:
{
  "accuracy": 5,
  "completeness": 4,
  "clarity": 5,
  "tone": 4,
  "overall": 4.5,
  "reasoning": "Response is accurate and clear. Could be more detailed about checking spam folder."
}
""")

## 📊 Comprehensive Model Evaluator

In [ ]:
class ModelEvaluator:
    """Comprehensive model evaluation."""
    
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    def evaluate_on_test_set(
        self,
        test_set: List[Dict],
        system_prompt: str = None
    ) -> Dict:
        """Run model on test set and calculate metrics."""
        
        results = []
        
        for example in test_set:
            # Generate prediction
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": example['input']})
            
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=messages,
                temperature=0.7
            )
            
            prediction = response.choices[0].message.content
            
            # Calculate simple overlap
            expected_words = set(example['expected_output'].lower().split())
            pred_words = set(prediction.lower().split())
            overlap = len(expected_words & pred_words) / len(expected_words) if expected_words else 0
            
            results.append({
                'input': example['input'],
                'expected': example['expected_output'],
                'prediction': prediction,
                'category': example['category'],
                'word_overlap': overlap,
                'length': len(prediction),
            })
        
        # Aggregate metrics
        avg_overlap = np.mean([r['word_overlap'] for r in results])
        avg_length = np.mean([r['length'] for r in results])
        
        # By category
        by_category = defaultdict(list)
        for r in results:
            by_category[r['category']].append(r['word_overlap'])
        
        category_scores = {
            cat: np.mean(scores) for cat, scores in by_category.items()
        }
        
        return {
            'model': self.model_name,
            'num_examples': len(results),
            'avg_word_overlap': avg_overlap,
            'avg_length': avg_length,
            'by_category': category_scores,
            'results': results
        }

# Example usage
print("📊 Model Evaluator Example\n")
print("""
# Evaluate base model
base_evaluator = ModelEvaluator("gpt-3.5-turbo")
base_results = base_evaluator.evaluate_on_test_set(
    test_set,
    system_prompt="You are a helpful customer support agent."
)

# Evaluate fine-tuned model
ft_evaluator = ModelEvaluator("ft:gpt-3.5-turbo:company:v1:abc")
ft_results = ft_evaluator.evaluate_on_test_set(test_set)

# Compare
print(f"Base model overlap: {base_results['avg_word_overlap']:.2%}")
print(f"Fine-tuned overlap: {ft_results['avg_word_overlap']:.2%}")
""")

## 🔍 Detecting Overfitting

In [ ]:
def check_overfitting(
    train_accuracy: float,
    val_accuracy: float,
    threshold: float = 0.1
) -> Dict:
    """Check for overfitting."""
    
    gap = train_accuracy - val_accuracy
    
    if gap > threshold:
        status = "⚠️  Overfitting detected"
        recommendation = "Try: reduce epochs, increase data, add regularization"
    elif gap < -threshold:
        status = "⚠️  Underfitting detected"
        recommendation = "Try: increase epochs, increase model capacity, better data quality"
    else:
        status = "✅ Good fit"
        recommendation = "Model is well-balanced"
    
    return {
        'status': status,
        'train_accuracy': train_accuracy,
        'val_accuracy': val_accuracy,
        'gap': gap,
        'recommendation': recommendation
    }

# Examples
print("🔍 Overfitting Detection\n")
print("="*60)

scenarios = [
    {"train": 0.95, "val": 0.92, "name": "Good fit"},
    {"train": 0.98, "val": 0.75, "name": "Overfitting"},
    {"train": 0.65, "val": 0.70, "name": "Underfitting"},
]

for scenario in scenarios:
    result = check_overfitting(scenario['train'], scenario['val'])
    print(f"\n{scenario['name']}:")
    print(f"  Train: {result['train_accuracy']:.1%}")
    print(f"  Val:   {result['val_accuracy']:.1%}")
    print(f"  Gap:   {result['gap']:.1%}")
    print(f"  {result['status']}")
    print(f"  → {result['recommendation']}")

## 🆚 A/B Testing Framework

In [ ]:
class ABTest:
    """A/B test two models."""
    
    def __init__(self, model_a: str, model_b: str):
        self.model_a = model_a
        self.model_b = model_b
        self.results = []
    
    def run_comparison(
        self,
        test_cases: List[Dict],
        system_prompt: str = None
    ) -> Dict:
        """Compare models on test cases."""
        
        print(f"🆚 Comparing {self.model_a} vs {self.model_b}\n")
        
        for i, test_case in enumerate(test_cases, 1):
            print(f"Test {i}/{len(test_cases)}: {test_case['input'][:50]}...")
            
            # Get responses from both models
            # (Simulated for demo)
            
            self.results.append({
                'input': test_case['input'],
                'response_a': "Response from model A",
                'response_b': "Response from model B",
                'expected': test_case.get('expected_output'),
            })
        
        return self.analyze_results()
    
    def analyze_results(self) -> Dict:
        """Analyze A/B test results."""
        
        # In production, calculate actual metrics
        return {
            'model_a': self.model_a,
            'model_b': self.model_b,
            'num_tests': len(self.results),
            'winner': 'Model B',  # Simulated
            'confidence': 0.85,
        }

# Example
print("🆚 A/B Testing Example\n")
print("""
# Setup test
ab_test = ABTest(
    model_a="gpt-3.5-turbo",
    model_b="ft:gpt-3.5-turbo:company:v1:abc"
)

# Run comparison
results = ab_test.run_comparison(test_set)

# Analyze
if results['winner'] == 'Model B' and results['confidence'] > 0.8:
    print("✅ Fine-tuned model is better! Deploy it.")
else:
    print("⚠️  Not enough improvement. Keep base model.")
""")

## ✅ Summary

### Evaluation Checklist:

**1. Create Quality Test Set**
```python
# Requirements:
- 50-100 examples minimum
- Cover all use cases
- Include edge cases
- Never use in training!
- Have expected outputs
```

**2. Quantitative Metrics**
```python
# Accuracy metrics
- Exact match (strict)
- Word overlap (lenient)
- BLEU/ROUGE (generation)
- F1 score (classification)
```

**3. Qualitative Evaluation**
```python
# LLM-as-Judge
- Accuracy
- Completeness
- Clarity
- Tone/style
- Helpfulness
```

**4. Overfitting Detection**
```python
if train_acc - val_acc > 0.1:
    # Overfitting!
    # Solutions:
    - Reduce epochs
    - Add more diverse data
    - Increase regularization
    - Early stopping
```

### Evaluation Workflow:

```python
# 1. Split data
train, val, test = split_data(data, [0.7, 0.15, 0.15])

# 2. Train model
model = train_model(train)

# 3. Validate (catch overfitting)
val_metrics = evaluate(model, val)
check_overfitting(train_metrics, val_metrics)

# 4. Final test (unbiased evaluation)
test_metrics = evaluate(model, test)

# 5. A/B test in production
ab_test(base_model, fine_tuned_model)
```

### Key Metrics:

**For Classification:**
```python
Accuracy = (TP + TN) / Total
Precision = TP / (TP + FP)
Recall = TP / (TP + FN)
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```

**For Generation:**
```python
Word Overlap = |pred_words ∩ expected_words| / |expected_words|
BLEU = measure of n-gram overlap
ROUGE = measure of recall
```

### When to Deploy:

✅ **Deploy fine-tuned model if:**
- Test set accuracy > base model + 5%
- No overfitting (train-val gap < 10%)
- A/B test shows improvement
- Passes human review

❌ **Keep base model if:**
- Minimal improvement (< 3%)
- Overfitting detected
- Worse on edge cases
- Inconsistent responses

### Continuous Evaluation:

```python
# Monitor in production
class ProductionMonitor:
    def log_interaction(self, query, response, user_feedback):
        # Track metrics
        - Response quality (user ratings)
        - Task completion
        - Response time
        - Cost per request
        
    def detect_drift(self):
        # Alert if:
        - Accuracy drops > 5%
        - User satisfaction drops
        - New query patterns emerge
        
    def trigger_retrain(self):
        # Retrain when:
        - Drift detected
        - New data available
        - Quarterly schedule
```

### Evaluation Tools:

**1. Automated Metrics**
- Fast, cheap
- Use for continuous monitoring
- Good proxy for quality

**2. LLM-as-Judge**
- More nuanced
- Catches subtle issues
- Use for validation

**3. Human Evaluation**
- Most accurate
- Expensive, slow
- Use for final validation
- Sample 5-10% of responses

### Best Practices:

1. **Never test on training data**
   - Always use held-out test set
   - Ideally: train/val/test split

2. **Track everything**
   ```python
   # Log all evaluations
   - Model version
   - Test set version
   - Timestamp
   - All metrics
   ```

3. **Automate evaluation**
   ```python
   # CI/CD pipeline
   on_new_model:
     - Run full test suite
     - Compare to baseline
     - Alert if regression
   ```

4. **Set quality thresholds**
   ```python
   THRESHOLDS = {
       'accuracy': 0.85,
       'train_val_gap': 0.1,
       'user_satisfaction': 4.0,
   }
   ```

### Next: `06_fine_tuning/06_hyperparameters.ipynb`